# Synthetic Spike Injection — Visual Check (Train & Test)

Standalone diagnostic notebook. Loads `kagglephase1.ipynb`'s saved output
(`features_injected.npy`, `features_clean.npy`, `spike_mask.npy`,
`spike_events.json`, `segments.json`) and plots clean vs. injected values
overlaid, with burst event windows shaded, separately for the TRAIN region
and the TEST region of each container -- so you can see exactly what the
synthetic bursts look like and confirm they landed where the design intended.

Fully self-contained: reads directly from Phase 1's saved files on disk (not
from any other notebook's live session state), and does not modify
`kagglephase1.ipynb`, `kagglephase2.ipynb`, `kagglephase3.ipynb`, or any
other existing file.

**Requires:** the `kagglephase1-output` dataset attached via Add Input (the
same one produced by `kagglephase1.ipynb` and already used by Phase 2/3).


## Step 1: Locate Phase 1's Output

In [ ]:
import os, json
import numpy as np
from pathlib import Path

IN_KAGGLE = os.path.exists('/kaggle')
search_root = Path('/kaggle/input') if IN_KAGGLE else Path('.')

hits = sorted(search_root.glob('**/features_injected.npy'), key=lambda p: len(p.parts))
if not hits:
    raise FileNotFoundError(
        f"features_injected.npy not found under {search_root} -- attach the "
        f"'kagglephase1-output' dataset via 'Add Input' (the output of kagglephase1.ipynb), "
        f"then re-run this cell."
    )
P1 = hits[0].parent
print(f"Phase-1 output found: {P1}")
for name in ('features_injected.npy', 'features_clean.npy', 'spike_mask.npy',
             'spike_events.json', 'segments.json', 'feature_cols.json',
             'normalization_stats.json'):
    p = P1 / name
    print(f"  {name:<28} exists={p.exists()}")
    if not p.exists():
        raise FileNotFoundError(f"{p} missing -- re-download a complete kagglephase1_output.zip.")


## Step 2: Load Everything and Convert to Real Units

In [ ]:
feat_inj = np.load(P1 / 'features_injected.npy', mmap_mode='r')
feat_cln = np.load(P1 / 'features_clean.npy', mmap_mode='r')
spike_mask = np.load(P1 / 'spike_mask.npy')
events = json.load(open(P1 / 'spike_events.json'))
segments = json.load(open(P1 / 'segments.json'))
fc = json.load(open(P1 / 'feature_cols.json'))
stats = json.load(open(P1 / 'normalization_stats.json'))

TARGET_COLUMNS = fc['target_columns']
TARGET_NAMES = fc['target_names']
TARGET_IDX = fc['target_idx']
TARGET_MEAN = np.array([stats[c]['mean'] for c in TARGET_COLUMNS])
TARGET_STD = np.array([stats[c]['std'] for c in TARGET_COLUMNS])

def real_units(feat_arr, row_slice):
    norm = np.asarray(feat_arr[row_slice])[:, TARGET_IDX]
    return norm * TARGET_STD + TARGET_MEAN

print(f"features shape: {feat_inj.shape}  ({feat_inj.shape[0]:,} rows, {feat_inj.shape[1]} features)")
print(f"target columns: {dict(zip(TARGET_NAMES, TARGET_IDX))}")
print(f"containers: {len(segments)}")
by_split = {}
for e in events:
    by_split.setdefault(e['split'], []).append(e)
print(f"burst events: { {k: len(v) for k, v in by_split.items()} }")
print(f"rows inside burst regions: {spike_mask.sum():,} / {len(spike_mask):,} ({spike_mask.mean()*100:.2f}%)")


## Step 3: Visualize TRAIN Region — Clean vs. Injected, Events Shaded

One figure per sample container, all 4 targets, restricted to that
container's TRAIN rows only (`segments[cid]['start']` to `segments[cid]['i70']`).
Shaded bands mark each injected event's `[start, end)` range.


In [ ]:
import matplotlib.pyplot as plt

def plot_region(cid, row_lo, row_hi, region_name, container_events):
    sl = slice(row_lo, row_hi)
    clean_real = real_units(feat_cln, sl)
    inj_real = real_units(feat_inj, sl)
    x = np.arange(row_lo, row_hi)

    fig, axes = plt.subplots(len(TARGET_NAMES), 1, figsize=(14, 11), sharex=True)
    for i, name in enumerate(TARGET_NAMES):
        axes[i].plot(x, clean_real[:, i], label='clean', color='tab:blue', linewidth=0.9)
        axes[i].plot(x, inj_real[:, i], label='injected', color='tab:red', linewidth=0.9, alpha=0.8)
        for e in container_events:
            if e['start'] < row_hi and e['end'] > row_lo:
                axes[i].axvspan(max(e['start'], row_lo), min(e['end'], row_hi),
                                color='orange', alpha=0.2)
        axes[i].set_title(f'{name} -- {cid} ({region_name})')
        axes[i].set_ylabel(name)
        axes[i].legend(fontsize=7, loc='upper right')
        axes[i].grid(alpha=0.3)
    axes[-1].set_xlabel('row index')
    plt.tight_layout()
    save_dir = Path('/kaggle/working') if IN_KAGGLE else Path('.')
    save_path = save_dir / f'spike_injection_{cid}_{region_name}.png'
    plt.savefig(save_path, dpi=100)
    plt.show()
    print(f"Saved {save_path}  ({len(container_events)} event(s) in view)")

sample_containers = list(segments.keys())[:3]
for cid in sample_containers:
    seg = segments[cid]
    c_events = [e for e in events if e['container'] == cid and e['split'] == 'train']
    plot_region(cid, seg['start'], seg['i70'], 'TRAIN', c_events)


## Step 4: Visualize TEST Region — Clean vs. Injected, Events Shaded

Same plot, restricted to each container's TEST rows
(`segments[cid]['i85']` to `segments[cid]['end']`).


In [ ]:
for cid in sample_containers:
    seg = segments[cid]
    c_events = [e for e in events if e['container'] == cid and e['split'] == 'test']
    plot_region(cid, seg['i85'], seg['end'], 'TEST', c_events)


## Step 5: Event Magnitude Summary (Train vs. Val vs. Test)

In [ ]:
print(f"{'Split':<8} {'n_events':>9} {'cpu peak_extra_rate (mean)':>28} {'mem peak (mean, % of level)':>30}")
print("-" * 80)
for split_name in ('train', 'val', 'test'):
    evs = [e for e in events if e['split'] == split_name]
    if not evs:
        print(f"{split_name:<8} {0:>9}")
        continue
    cpu_col = TARGET_COLUMNS[0]
    cpu_rates = [e['targets'][cpu_col]['peak_extra_rate'] for e in evs if cpu_col in e['targets']]
    mem_cols = [c for c in TARGET_COLUMNS if c != cpu_col]
    mem_peaks = []
    for e in evs:
        for mc in mem_cols:
            if mc in e['targets']:
                mem_peaks.append(e['targets'][mc]['peak'])
    print(f"{split_name:<8} {len(evs):>9} {np.mean(cpu_rates):>28.3f} {np.mean(mem_peaks):>30.1f}")

print()
print("Peak values match the design intent from kagglephase1.ipynb Step 5: cpu bursts are")
print("rate-bumps (permanent counter shift), memory bursts are bump-and-return, magnitudes")
print("anchored to each container's own p95 natural step / median level.")
